# Week 6 · Try It: Position a Phone-Grade GNSS Receiver in Tsim Sha Tsui

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%206/w6_gnss_positioning.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

Real pseudoranges from a u-blox ZED-F9P receiver on a car driving through Tsim Sha Tsui on 17 May 2021 (UrbanNav-HK-Medium-Urban-1), with the car's true position from a NovAtel SPAN-CPT GNSS/INS system. The receiver tracks GPS, QZSS, Galileo and BeiDou.

The method is the one in Section 5 of the slides: Gauss–Newton on

$$r_i = \|\mathbf{s}_i - \mathbf{p}\| + b - \rho_i \quad\text{(predicted − measured)}, \qquad \mathbf{J}_i = [\,\mathbf{u}_i^{\mathsf T},\ 1\,], \qquad \mathbf{u}_i = \frac{\mathbf{p} - \mathbf{s}_i}{\|\mathbf{p} - \mathbf{s}_i\|},$$

$$\Delta\mathbf{x} = -(\mathbf{J}^{\mathsf T}\mathbf{W}\mathbf{J})^{-1}\mathbf{J}^{\mathsf T}\mathbf{W}\mathbf{r}.$$

Each satellite system has its own receiver clock $b$, so with three groups (GPS+QZSS, Galileo, BeiDou) there are 3 + 3 = 6 unknowns per epoch.

**Questions** (the same as on the slide)
- (a) Median horizontal error with plain least squares?
- (b) And with the weights $w_i = \sin^2(\text{elevation}) \cdot 10^{(C/N_0 - 45)/10}$?
- (c) How many iterations are needed from the centre of the Earth?
- (d) Drop every satellite with $C/N_0$ < 30 dB-Hz instead of weighting. What happens?

## Part 0 · Load the data

One row per epoch and satellite. `sat_x, sat_y, sat_z`: satellite position (ECEF, m) at the time of transmission, already rotated for the Earth's turn during the signal's flight. `pr_corr`: the pseudorange with the satellite clock, ionosphere and troposphere corrections already applied, so `pr_corr` ≈ range + receiver clock + error. `el_deg`: elevation; `cn0`: signal strength (dB-Hz). Satellites below 10° are already removed.

In [ ]:
import os, urllib.request
URL = "https://raw.githubusercontent.com/weisongwen/AAE4011-S22627/main/Lecture%20Notes/Week%206/"
for f in ['w6_tst_gnss.csv.gz', 'w6_tst_truth.csv.gz']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
print("ready:", {f: os.path.getsize(f) // 1024 for f in ['w6_tst_gnss.csv.gz', 'w6_tst_truth.csv.gz']}, "KB")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

obs = pd.read_csv("w6_tst_gnss.csv.gz")
truth = pd.read_csv("w6_tst_truth.csv.gz").set_index("epoch")
n_sat = obs.groupby("epoch").size()
print(len(obs), "rows,", obs.epoch.nunique(), "epochs,", truth.shape[0], "true positions")
print("satellites per epoch: median", int(n_sat.median()), " min", n_sat.min(), " max", n_sat.max())
print("rows per system:", obs.sys.value_counts().to_dict())
t = np.sort(obs.gps_sow.unique())
print(f"span {t[-1] - t[0]:.0f} s; largest gap between epochs {np.diff(t).max():.0f} s")
obs.head()

The receiver log has one gap of 131 s; otherwise there is one epoch per second.

## Part 1 · Gauss–Newton for one epoch

Fill in the two lines marked TODO, then run the check below.

- TODO 1: the residual vector $\mathbf{r}$, predicted minus measured (use `pred` and `pr`).
- TODO 2: the first three columns of $\mathbf{J}$: the unit vectors $\mathbf{u}_i$ from the satellites to the receiver (use `x[:3]`, `sats` and `d`).

In [ ]:
CLOCK = {"G": 0, "J": 0, "E": 1, "C": 2}      # one receiver clock per system; QZSS (J) uses GPS time
X0 = (-2418000.0, 5386000.0, 2405000.0)        # a start in Hong Kong (ECEF, m)


def gauss_newton(sats, pr, systems, w=None, x0=X0, iters=20, tol=1e-4):
    """Weighted Gauss-Newton for one epoch.
    sats: (m, 3) satellite positions, pr: (m,) corrected pseudoranges, systems: m letters G/J/E/C, w: (m,) weights.
    Unknowns: receiver x, y, z (ECEF, m), then one clock bias b (m) per system group present."""
    groups = sorted({CLOCK[s] for s in systems})
    col = {g: 3 + i for i, g in enumerate(groups)}        # column of each clock in J
    x = np.zeros(3 + len(groups))
    x[:3] = x0
    w = np.ones(len(pr)) if w is None else np.asarray(w, float)
    for it in range(1, iters + 1):
        d = np.linalg.norm(sats - x[:3], axis=1)          # geometric ranges |s_i - p|
        b = np.array([x[col[CLOCK[s]]] for s in systems]) # the clock of each satellite's system
        pred = d + b                                      # predicted pseudoranges
        r = ...                                    # TODO 1: residuals, predicted minus measured
        J = np.zeros((len(pr), len(x)))
        J[:, :3] = ...                             # TODO 2: unit vectors from the satellites to the receiver
        for i, s in enumerate(systems):
            J[i, col[CLOCK[s]]] = 1.0                     # d r_i / d b = 1 for the satellite's own clock
        dx = -np.linalg.solve(J.T @ (w[:, None] * J), J.T @ (w * r))
        x = x + dx
        if np.linalg.norm(dx) < tol:
            break
    return x, it, col

In [ ]:
def enu_matrix(p):
    """Rows: east, north, up unit vectors (WGS 84) at the ECEF point p."""
    x, y, z = p
    lon = np.arctan2(y, x)
    e2 = 6.69437999014e-3
    lat = np.arctan2(z, np.hypot(x, y) * (1 - e2))
    for _ in range(5):
        N = 6378137.0 / np.sqrt(1 - e2 * np.sin(lat) ** 2)
        h = np.hypot(x, y) / np.cos(lat) - N
        lat = np.arctan2(z, np.hypot(x, y) * (1 - e2 * N / (N + h)))
    sl, cl, so, co = np.sin(lat), np.cos(lat), np.sin(lon), np.cos(lon)
    return np.array([[-so, co, 0], [-sl * co, -sl * so, cl], [cl * co, cl * so, sl]])

In [ ]:
# check your two lines on the first epoch
g = obs[obs.epoch == 0]
REFERENCE = np.array([-2418219.939, 5385992.515, 2405321.881])   # ECEF (m)
try:
    x, it, col = gauss_newton(g[["sat_x", "sat_y", "sat_z"]].to_numpy(), g.pr_corr.to_numpy(), list(g.sys))
    if np.linalg.norm(x[:3] - REFERENCE) < 0.01:
        p = truth.loc[0, ["x", "y", "z"]].to_numpy(float)
        e, n, u = enu_matrix(p) @ (x[:3] - p)
        print(f"epoch 0 correct: {it} iterations, {len(g)} satellites, horizontal error "
              f"{np.hypot(e, n):.1f} m (east {e:+.1f}, north {n:+.1f})")
    else:
        print("not yet: the answer is", round(float(np.linalg.norm(x[:3] - REFERENCE))), "m from the reference. Check the signs.")
except TypeError:
    print("Fill in TODO 1 and TODO 2 first.")

## Part 2 · All epochs, plain least squares → question (a)

`solve_all` runs Gauss–Newton on every epoch and compares with the truth in east, north and up. The optional `min_cn0` drops weak satellites, and an epoch is only solved when it keeps at least one more satellite than unknowns (so that there is a residual left to check).

In [ ]:
def solve_all(weights=None, x0=X0, min_cn0=None):
    rows = []
    for k, g in obs.groupby("epoch"):
        if min_cn0 is not None:
            g = g[g.cn0 >= min_cn0]
        systems = list(g.sys)
        n_unknowns = 3 + len({CLOCK[s] for s in systems})
        if len(g) < n_unknowns + 1:
            rows.append(dict(epoch=k, solved=False))
            continue
        w = None if weights is None else weights(g)
        x, it, col = gauss_newton(g[["sat_x", "sat_y", "sat_z"]].to_numpy(), g.pr_corr.to_numpy(),
                                  systems, w, x0)
        p = truth.loc[k, ["x", "y", "z"]].to_numpy(float)
        e, n, u = enu_matrix(p) @ (x[:3] - p)
        rows.append(dict(epoch=k, solved=True, t=g.gps_sow.iloc[0], X=x[0], Y=x[1], Z=x[2], east=e,
                         north=n, up=u, horiz=np.hypot(e, n), iters=it, n_sat=len(g),
                         clock_gps=x[col[0]] if 0 in col else np.nan))
    return pd.DataFrame(rows).set_index("epoch")


ls = solve_all()
print(f"(a) least squares: median horizontal error {ls.horiz.median():.1f} m, "
      f"95 % below {ls.horiz.quantile(0.95):.1f} m, median {ls.iters.median():.0f} iterations")
print(f"GPS receiver clock: median {ls.clock_gps.median() / 1000:.0f} km "
      f"= {ls.clock_gps.median() / 299792458 * 1000:.2f} ms")

The receiver clock is about 5 ms off GPS time: 1500 km of range. That is why $b$ must be an unknown.

## Part 3 · Weighted least squares → question (b)

Low satellites and weak signals are the ones most likely to be reflected by buildings (NLOS) or noisy, so they get small weights.

In [ ]:
def weights(g):
    return np.sin(np.radians(g.el_deg.to_numpy())) ** 2 * 10 ** ((g.cn0.to_numpy() - 45) / 10)


wls = solve_all(weights)
print(f"(b) weighted:      median horizontal error {wls.horiz.median():.1f} m, "
      f"95 % below {wls.horiz.quantile(0.95):.1f} m")

In [ ]:
t0 = ls.t.min()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(ls.t - t0, ls.horiz, ".", ms=3, label=f"least squares, median {ls.horiz.median():.1f} m")
ax[0].plot(wls.t - t0, wls.horiz, ".", ms=3, label=f"weighted, median {wls.horiz.median():.1f} m")
ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("horizontal error (m)"); ax[0].set_ylim(0, 60)
ax[0].legend(); ax[0].set_title("Error against the truth")

o = truth.loc[0, ["x", "y", "z"]].to_numpy(float)
R = enu_matrix(o)
def to_en(df, cols=("X", "Y", "Z")):
    return (R @ (df[list(cols)].to_numpy(float) - o).T)[:2]
tr = to_en(truth.loc[ls.index], ("x", "y", "z"))
tr[:, np.r_[False, np.diff(ls.t.to_numpy()) > 1.5]] = np.nan      # break the line at the 131 s gap
ax[1].plot(*tr, "k-", lw=2, label="truth (SPAN-CPT)")
ax[1].plot(*to_en(ls), ".", ms=2, label="least squares")
ax[1].plot(*to_en(wls), ".", ms=2, label="weighted")
ax[1].set_aspect("equal"); ax[1].set_xlabel("east (m)"); ax[1].set_ylabel("north (m)")
ax[1].legend(); ax[1].set_title("Track in Tsim Sha Tsui")
plt.tight_layout(); plt.show()

## Part 4 · Start from the centre of the Earth → question (c)

In [ ]:
centre = solve_all(weights, x0=(0.0, 0.0, 0.0))
print(f"(c) from the centre of the Earth: median {centre.iters.median():.0f} iterations, "
      f"at most {centre.iters.max()}; from Hong Kong: median {wls.iters.median():.0f}")
print(f"    same answers as before: largest difference "
      f"{np.max(np.abs(centre[['X', 'Y', 'Z']].to_numpy() - wls[['X', 'Y', 'Z']].to_numpy())):.1e} m")

## Part 5 · Exclude weak signals instead of weighting → question (d)

In [ ]:
excl = solve_all(min_cn0=30)
ok = excl[excl.solved]
print(f"(d) C/N0 >= 30 dB-Hz only: median {ok.horiz.median():.1f} m on the {len(ok)} epochs still solvable;"
      f" {len(excl) - len(ok)} of {len(excl)} epochs no longer have enough satellites")
print(f"    weighted least squares on those same {len(ok)} epochs: median "
      f"{wls.loc[ok.index].horiz.median():.1f} m")

plt.figure(figsize=(6, 3.5))
plt.hist(obs.cn0, bins=np.arange(10, 56, 1), color="0.6")
plt.axvline(30, color="r"); plt.xlabel("C/N0 (dB-Hz)"); plt.ylabel("satellite rows")
plt.title("Signal strength in the street canyon"); plt.show()

---
## Check your answers (try first!)

- TODO 1: `r = pred - pr`. TODO 2: `J[:, :3] = (x[:3] - sats) / d[:, None]`.
- (a) 10.6 m (95 % of epochs below 46.5 m).
- (b) 5.2 m (95 % below 19.0 m): the weights trust the high, strong satellites, which are the ones least likely to be reflected.
- (c) About 5 iterations (at most 6), against 3 from a start in Hong Kong, with the same answers. From 20 000 km away, the range is almost linear in the position, so Gauss–Newton hardly needs a good start.
- (d) Median 3.9 m on the 512 epochs that are still solvable, but 145 of 657 epochs no longer have enough satellites. On those same 512 epochs, weighting gives 4.2 m, so excluding helps a little where it works, but it loses one epoch in five. In the Week 8 lab you will classify each satellite as line-of-sight or not, then exclude or down-weight it.

**Data.** UrbanNav dataset (github.com/IPNL-POLYU/UrbanNavDataset), Intelligent Positioning and Navigation Laboratory, The Hong Kong Polytechnic University, MIT licence. Satellite orbits and clocks: IGS merged broadcast ephemeris BRDM (DLR), from the IGS/BKG data centre. The two CSV files were made from these for this course: broadcast satellite positions and clocks, Klobuchar ionosphere and Saastamoinen troposphere corrections, 10° elevation mask.